# Comparative Results Summary

This notebook loads `experiments/full_evaluation.json` and creates a compact comparison table for all saved models.

It focuses on the most relevant metrics for the fraud detection project:
- PR-AUC
- F1
- Recall
- NLL
- Brier Score
- ECE


In [6]:
from pathlib import Path
import json
import pandas as pd

RESULTS_PATH = Path('experiments/full_evaluation.json')

if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        f'Could not find results file at: {RESULTS_PATH}\n'
        'Run the evaluation script first, or update RESULTS_PATH.'
    )

with RESULTS_PATH.open('r', encoding='utf-8') as f:
    results = json.load(f)

print('Loaded:', RESULTS_PATH)


Loaded: experiments\full_evaluation.json


In [7]:
rows = []

for model_name, model_info in results['models'].items():
    for split_name in ['validation', 'test']:
        split_info = model_info[split_name]
        cls = split_info['classification_metrics']
        prob = split_info['proper_scoring_rules']
        cal = split_info['calibration_metrics']

        rows.append({
            'model': model_name,
            'split': split_name,
            'pr_auc': cls['pr_auc'],
            'f1': cls['f1'],
            'recall': cls['recall'],
            'precision': cls['precision'],
            'roc_auc': cls['roc_auc'],
            'nll': prob['nll'],
            'brier_score': prob['brier_score'],
            'ece': cal['ece'],
            'mce': cal['mce'],
        })

df = pd.DataFrame(rows)
df.head()


,model,split,pr_auc,f1,recall,precision,roc_auc,nll,brier_score,ece,mce
0,baseline_results/logistic_regression,validation,0.680711,0.111879,0.898990,0.059651,0.974728,0.101567,0.022154,0.064351,0.838180
1,baseline_results/logistic_regression,test,0.717874,0.115533,0.918367,0.061644,0.972552,0.104314,0.022053,0.064444,0.850722
2,baseline_results/random_forest,validation,0.811919,0.807018,0.696970,0.958333,0.946509,0.007689,0.000539,0.000301,0.447143
3,baseline_results/random_forest,test,0.851120,0.832370,0.734694,0.960000,0.956872,0.006169,0.000434,0.000293,0.512917
4,boosting_results/lightgbm,validation,0.049408,0.110661,0.828283,0.059291,0.902679,0.799185,0.023138,0.023138,0.940709


In [8]:
# Compact table focused on the test split
test_df = df[df['split'] == 'test'].copy()
test_df = test_df.sort_values(by=['pr_auc', 'f1', 'nll'], ascending=[False, False, True])

compact_cols = [
    'model', 'pr_auc', 'f1', 'recall', 'precision', 'nll', 'brier_score', 'ece'
]

compact_table = test_df[compact_cols].reset_index(drop=True)
compact_table.style.format({
    'pr_auc': '{:.4f}',
    'f1': '{:.4f}',
    'recall': '{:.4f}',
    'precision': '{:.4f}',
    'nll': '{:.4f}',
    'brier_score': '{:.4f}',
    'ece': '{:.4f}',
})


,model,pr_auc,f1,recall,precision,nll,brier_score,ece
0,boosting_results/xgboost,0.8697,0.8400,0.8571,0.8235,0.0031,0.0005,0.0006
1,baseline_results/random_forest,0.8511,0.8324,0.7347,0.9600,0.0062,0.0004,0.0003
2,baseline_results/logistic_regression,0.7179,0.1155,0.9184,0.0616,0.1043,0.0221,0.0644
3,bnn_results/bayesian_neural_network,0.7151,0.1154,0.9082,0.0616,0.1401,0.0297,0.1147
4,boosting_results/lightgbm,0.0512,0.1122,0.8469,0.0601,0.7962,0.0231,0.0231


In [9]:
# Validation vs test side by side
pivot_table = df.pivot(index='model', columns='split', values=['pr_auc', 'f1', 'recall', 'nll', 'brier_score', 'ece'])
pivot_table = pivot_table.sort_values(by=('pr_auc', 'test'), ascending=False)
pivot_table.style.format('{:.4f}')


In [10]:
# Optional: export compact table to CSV
output_csv = Path('experiments/model_comparison_table.csv')
compact_table.to_csv(output_csv, index=False)
print(f'Saved compact comparison table to: {output_csv}')


Saved compact comparison table to: experiments\model_comparison_table.csv
